In [1]:
import pandas as pd
import numpy as np
import os
import glob
import re
from huggingface_hub import snapshot_download
from transformers import TimesFmModel, TimesFmConfig
import torch
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from collections import Counter
from collections import defaultdict
import hdbscan
from sklearn.cluster import DBSCAN
from k_means_constrained import KMeansConstrained
from sklearn.cluster import SpectralClustering

# pip install torch torch-geometric
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.neighbors import kneighbors_graph

from chronos import ChronosPipeline


/opt/miniconda3/envs/cronos/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_dir = ".././Datasets/Labeled_Data_Without_GPS/USA/"
file_paths = glob.glob(os.path.join(data_dir, "**", "*.csv"), recursive=True)

files_df = pd.DataFrame({
    "full_path": file_paths,
    "filename": [os.path.basename(p) for p in file_paths],
})

files_df["surface_id"] = files_df["filename"].str.extract(r"SurfaceTypeID_(\d+)").astype("Int64")

files_df.head(2) 

,full_path,filename,surface_id
0,.././Datasets/Labeled_Data_Without_GPS/USA/Sur...,2019-02-27_SurfaceTypeID_9_SamsungGalaxyJ7_exp...,9
1,.././Datasets/Labeled_Data_Without_GPS/USA/Sur...,2019-09-02_SurfaceTypeID_9_SamsungGalaxyS7_exp...,9


In [3]:
from tqdm import tqdm

# Windowing rows into fixed-size segments for model input

window_size = 416
step_percentage = 0.5
step = int(window_size * step_percentage)


windows = []
labels = []

for _, row in tqdm(files_df.iterrows(), total=len(files_df), desc="Processing files"):
    path = row["full_path"]
    sid = row["surface_id"]
    df_current = pd.read_csv(path)

    # Filter for accelerometer data
    acc_df = df_current[df_current["sensorName"] == "Accelerometer"]

    if acc_df.shape[0] < window_size:
        continue
    
    try:
        acc_vals = acc_df[["valueX", "valueY", "valueZ"]].to_numpy()
        magnitude = np.linalg.norm(acc_vals, axis=1)
    except KeyError:
        print(f"Missing columns in {path}")
        continue
    
    for start in range(0, len(magnitude) - window_size + 1, step):
        window = magnitude[start:start + window_size]

        windows.append(window)
        labels.append(sid)

print(f"Total windows: {len(windows)}")

Processing files:  50%|████▉     | 320/645 [00:05<00:12, 25.26it/s] 

Missing columns in .././Datasets/Labeled_Data_Without_GPS/USA/SurfaceTypeID_8/2019-09-03_SurfaceTypeID_8_SamsungGalaxyJ7_exp6_subject2.csv


Processing files: 100%|██████████| 645/645 [00:13<00:00, 49.60it/s] 

Total windows: 67823


In [4]:
print(windows[0].shape)

(416,)


# Model part

In [5]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# Load Chronos — swap tag for chronos-t5-small / base / large as needed
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-tiny",
    device_map=device,
    torch_dtype=torch.float32,
)

# Access the underlying model for embedding extraction
model = pipeline.model
model.eval()

Using device: mps


`torch_dtype` is deprecated! Use `dtype` instead!


ChronosModel(
  (model): T5ForConditionalGeneration(
    (shared): Embedding(4096, 256)
    (encoder): T5Stack(
      (embed_tokens): Embedding(4096, 256)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=256, out_features=256, bias=False)
                (k): Linear(in_features=256, out_features=256, bias=False)
                (v): Linear(in_features=256, out_features=256, bias=False)
                (o): Linear(in_features=256, out_features=256, bias=False)
                (relative_attention_bias): Embedding(32, 4)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): T5LayerFF(
              (DenseReluDense): T5DenseActDense(
                (wi): Linear(in_features=256, out_features=1024, bias=False)
                (wo): Linear(in_features=

In [6]:
# Run this to see exact return signature
import inspect
print(inspect.signature(pipeline.model.encode))

# Also do a test call and check what comes back
batch_test = torch.stack([torch.tensor(windows[0], dtype=torch.float32)])
context_tensor, scale, _ = pipeline.tokenizer.context_input_transform(batch_test)
context_tensor = context_tensor.to(device)
scale = scale.to(device)

with torch.no_grad():
    result = pipeline.model.encode(context_tensor, scale)

print(type(result))
print(len(result) if isinstance(result, (tuple, list)) else "not a tuple")
if hasattr(result, '_fields'):  # namedtuple
    print(result._fields)

(input_ids: torch.Tensor, attention_mask: torch.Tensor)
<class 'torch.Tensor'>
not a tuple


In [7]:
def extract_embeddings_chronos(pipeline, windows, batch_size=32):
    
    all_embeddings = []

    for i in tqdm(range(0, len(windows), batch_size), desc="Extracting embeddings"):

        batch = windows[i:i + batch_size]

        batch_tensor = torch.stack([
            torch.tensor(w, dtype=torch.float32) for w in batch
        ])  # (B, T) — CPU for tokenization

        with torch.no_grad():

            context_tensor, scale, _ = pipeline.tokenizer.context_input_transform(
                batch_tensor
            )

            # Build correct attention mask: (B, T) matching context_tensor shape
            attention_mask = torch.ones(
                context_tensor.shape, dtype=torch.long
            )  # (B, T)

            context_tensor = context_tensor.to(device)
            attention_mask = attention_mask.to(device)

            embedding = pipeline.model.encode(
                input_ids=context_tensor,
                attention_mask=attention_mask
            )

        # embedding shape: (B, T_patches, hidden_dim)
        pooled = embedding.mean(dim=1)  # (B, hidden_dim)

        all_embeddings.append(pooled.detach().cpu().numpy())

    return np.vstack(all_embeddings)

In [9]:
embeddings = extract_embeddings_chronos(pipeline, windows, batch_size=32)
print("Embedding shape:", embeddings.shape)



Extracting embeddings:   1%|          | 15/2120 [00:05<11:48,  2.97it/s]


KeyboardInterrupt: 

# Clustering

In [10]:
labels = [l - 1 for l in labels]

In [11]:
from collections import Counter

label_counts = Counter(labels)

print("Label Distribution:")
for label in sorted(label_counts.keys()):
    count = label_counts[label]
    print(f"Label {label + 1}: {count} samples")

print(f"\nTotal samples: {sum(label_counts.values())}")

Label Distribution:
Label 1: 970 samples
Label 2: 8373 samples
Label 3: 5676 samples
Label 4: 5334 samples
Label 5: 9437 samples
Label 6: 10593 samples
Label 7: 11085 samples
Label 8: 10216 samples
Label 9: 2996 samples
Label 10: 3143 samples

Total samples: 67823


In [12]:
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

## K-means Clustering

In [13]:
num_clusters = len(np.unique(labels))

kmeans = KMeans(n_clusters=num_clusters, random_state=42)
clusters = kmeans.fit_predict(embeddings_scaled)

In [14]:
cluster_surface_map = {}

labels_array = np.array(labels)

for cluster_id in range(num_clusters):
    
    indices = np.where(clusters == cluster_id)[0]
    cluster_labels = labels_array[indices]
    
    dominant_surface = Counter(cluster_labels).most_common(1)[0][0]
    
    cluster_surface_map[cluster_id] = dominant_surface

print("\nCluster → Surface Mapping:")
for k, v in cluster_surface_map.items():
    print(f"Cluster {k} → Surface {v + 1}")


Cluster → Surface Mapping:
Cluster 0 → Surface 6
Cluster 1 → Surface 3
Cluster 2 → Surface 8
Cluster 3 → Surface 10
Cluster 4 → Surface 6
Cluster 5 → Surface 8
Cluster 6 → Surface 5
Cluster 7 → Surface 7
Cluster 8 → Surface 4
Cluster 9 → Surface 7


## HB Scan Clustering

In [15]:
# Create HDBSCAN model
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=10,      # minimum points to form a cluster
    min_samples=None,         # can tune for density sensitivity
    metric='euclidean'
)

# Fit and predict
clusters = clusterer.fit_predict(embeddings_scaled)

In [16]:
cluster_surface_map = {}

labels_array = np.array(labels)

# Get unique clusters (excluding noise = -1)
unique_clusters = [c for c in np.unique(clusters) if c != -1]

for cluster_id in unique_clusters:
    
    indices = np.where(clusters == cluster_id)[0]
    cluster_labels = labels_array[indices]
    
    # Skip empty clusters just in case
    if len(cluster_labels) == 0:
        continue
        
    dominant_surface = Counter(cluster_labels).most_common(1)[0][0]
    
    cluster_surface_map[cluster_id] = dominant_surface

print("\nCluster → Surface Mapping:")
for k, v in cluster_surface_map.items():
    print(f"Cluster {k} → Surface {v + 1}")

# Optional: count noise points
noise_count = np.sum(clusters == -1)
print(f"\nNoise points: {noise_count}")


Cluster → Surface Mapping:
Cluster 0 → Surface 7
Cluster 1 → Surface 10

Noise points: 969


## DB-Scan

In [17]:
from sklearn.cluster import DBSCAN

clusterer = DBSCAN(
    eps=0.5,              # neighborhood radius — tune this carefully
    min_samples=5,
    metric='euclidean'
)

clusters = clusterer.fit_predict(embeddings_scaled)

cluster_surface_map = {}
labels_array = np.array(labels)
unique_clusters = [c for c in np.unique(clusters) if c != -1]

for cluster_id in unique_clusters:
    indices = np.where(clusters == cluster_id)[0]
    cluster_labels = labels_array[indices]
    if len(cluster_labels) == 0:
        continue
    dominant_surface = Counter(cluster_labels).most_common(1)[0][0]
    cluster_surface_map[cluster_id] = dominant_surface

print("\nCluster → Surface Mapping:")
for k, v in cluster_surface_map.items():
    print(f"Cluster {k} → Surface {v + 1}")

noise_count = np.sum(clusters == -1)
print(f"\nNoise points: {noise_count}")


Cluster → Surface Mapping:
Cluster 0 → Surface 9
Cluster 1 → Surface 10
Cluster 2 → Surface 10
Cluster 3 → Surface 10

Noise points: 67792


## Balanced K-Means

In [ ]:
# pip install k-means-constrained
from k_means_constrained import KMeansConstrained

n_clusters = len(np.unique(labels))  # or set manually
size_min = len(embeddings_scaled) // (n_clusters * 2)
size_max = len(embeddings_scaled) // n_clusters * 2

clusterer = KMeansConstrained(
    n_clusters=n_clusters,
    size_min=size_min,
    size_max=size_max,
    random_state=42
)

clusters = clusterer.fit_predict(embeddings_scaled)

cluster_surface_map = {}
labels_array = np.array(labels)

for cluster_id in np.unique(clusters):
    indices = np.where(clusters == cluster_id)[0]
    cluster_labels = labels_array[indices]
    if len(cluster_labels) == 0:
        continue
    dominant_surface = Counter(cluster_labels).most_common(1)[0][0]
    cluster_surface_map[cluster_id] = dominant_surface

print("\nCluster → Surface Mapping:")
for k, v in cluster_surface_map.items():
    print(f"Cluster {k} → Surface {v + 1}")


Cluster → Surface Mapping:
Cluster 0 → Surface 7
Cluster 1 → Surface 5
Cluster 2 → Surface 6
Cluster 3 → Surface 2
Cluster 4 → Surface 6
Cluster 5 → Surface 8
Cluster 6 → Surface 5
Cluster 7 → Surface 8
Cluster 8 → Surface 7
Cluster 9 → Surface 4


: 

## Spectral Clustering


In [ ]:
from sklearn.cluster import SpectralClustering

n_clusters = len(np.unique(labels))  # or set manually

clusterer = SpectralClustering(
    n_clusters=n_clusters,
    affinity='rbf',          # 'nearest_neighbors' is faster for large data
    gamma=1.0,
    random_state=42,
    n_jobs=-1
)

clusters = clusterer.fit_predict(embeddings_scaled)

# No noise points in spectral clustering
cluster_surface_map = {}
labels_array = np.array(labels)

for cluster_id in np.unique(clusters):
    indices = np.where(clusters == cluster_id)[0]
    cluster_labels = labels_array[indices]
    if len(cluster_labels) == 0:
        continue
    dominant_surface = Counter(cluster_labels).most_common(1)[0][0]
    cluster_surface_map[cluster_id] = dominant_surface

print("\nCluster → Surface Mapping:")
for k, v in cluster_surface_map.items():
    print(f"Cluster {k} → Surface {v + 1}")

## GNN-based Clustering

In [ ]:
# pip install torch torch-geometric
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.neighbors import kneighbors_graph
from sklearn.cluster import KMeans

# --- Step 1: Build k-NN graph from embeddings ---
k = 10
A = kneighbors_graph(embeddings_scaled, k, mode='connectivity', include_self=False)
edge_index = torch.tensor(np.array(A.nonzero()), dtype=torch.long)
x = torch.tensor(embeddings_scaled, dtype=torch.float)

data = Data(x=x, edge_index=edge_index)

# --- Step 2: Define GCN encoder ---
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

# --- Step 3: Train with self-supervised loss (feature reconstruction) ---
in_dim = embeddings_scaled.shape[1]
model = GCNEncoder(in_dim, 64, 32)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

model.train()
for epoch in range(100):
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    # Reconstruction loss: encourage similar neighbors to have similar embeddings
    loss = F.mse_loss(out[edge_index[0]], out[edge_index[1]])
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# --- Step 4: Cluster GNN embeddings with KMeans ---
model.eval()
with torch.no_grad():
    gnn_embeddings = model(data.x, data.edge_index).numpy()

n_clusters = len(np.unique(labels))
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
clusters = kmeans.fit_predict(gnn_embeddings)

cluster_surface_map = {}
labels_array = np.array(labels)

for cluster_id in np.unique(clusters):
    indices = np.where(clusters == cluster_id)[0]
    cluster_labels = labels_array[indices]
    if len(cluster_labels) == 0:
        continue
    dominant_surface = Counter(cluster_labels).most_common(1)[0][0]
    cluster_surface_map[cluster_id] = dominant_surface

print("\nCluster → Surface Mapping:")
for k, v in cluster_surface_map.items():
    print(f"Cluster {k} → Surface {v + 1}")